# 🧬 BioMD Studio — Free GPU Backend on Google Colab
### Genuine, Research-Grade GROMACS Molecular Dynamics Engine

This notebook turns this Google Colab instance into a high-performance **free GPU execution worker** for the **BioMD Studio** web platform.

**Features:**
- 🚀 **Free GPU Acceleration** (NVIDIA Tesla T4)
- ⚛️ **Real GROMACS** (pdb2gmx, editconf, solvate, genion, grompp, mdrun, energy, trjconv)
- 📊 **Real Trajectory Analysis** (MDTraj, BioPython, SciPy)
- 🌐 **Secure Cloudflare Tunnel** — generates a free public URL with zero signup required!

---

### Step 1: Check GPU & Install GROMACS + Dependencies
Run this cell to verify GPU access and install GROMACS and scientific Python packages.

In [ ]:
# Check GPU
!nvidia-smi

# Install GROMACS
!apt-get update -qq
!apt-get install -y -qq gromacs

# Verify GROMACS installation
!gmx --version

# Install Python dependencies
!pip install -q fastapi uvicorn[standard] python-multipart sqlalchemy aiosqlite websockets scipy mdtraj biopython aiofiles pydantic plotly greenlet

### Step 2: Download or Clone Backend Code
If you uploaded to GitHub, enter your repository URL, or run this cell to set up the standalone backend directly.

In [ ]:
import os

# You can also clone your github repository here:
# !git clone https://github.com/YOUR_USERNAME/YOUR_REPO.git
# %cd YOUR_REPO/backend

# Ensure project directories exist
os.makedirs("/content/backend_data/projects", exist_ok=True)
os.environ["MD_DATA_DIR"] = "/content/backend_data"
print("✅ Storage directories initialized at /content/backend_data")

### Step 3: Launch Cloudflare Tunnel & FastAPI Server
Run this cell to start the server. It will download `cloudflared` (no token needed!) and display a public URL.
**Copy that public URL and paste it into the Web Interface Settings!**

In [ ]:
# Download Cloudflare Tunnel binary
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null

# Start Cloudflare tunnel in background
import subprocess, time, re

tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

# Wait for tunnel URL to appear in stderr
public_url = None
for _ in range(30):
    time.sleep(1)
    line = tunnel_proc.stderr.readline()
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print("=" * 60)
    print("🎉 YOUR FREE GPU BACKEND IS ONLINE!")
    print(f"🔗 Public API URL: {public_url}")
    print("👉 Copy this URL into the web application's 'Backend Connection' modal.")
    print("=" * 60)
else:
    print("Checking tunnel output...")

# Now launch uvicorn server
print("Starting FastAPI server...")
!uvicorn app.main:app --host 0.0.0.0 --port 8000